# Quick Start

[Learning Materials](README.md)

Run an A/B test, an A/A test, a homogeneity check, and a matching analysis on synthetic data. The pandas examples share the data in the first code cell. The Spark section can run on its own.

Each row represents one user. `Treatment` indicates their group (0 for control, 1 for treatment). `Target` is the outcome after treatment. `Target_lag_1` and `X_lag_1` were measured before treatment.

## Contents

- [A/B test](#quick-start-ab)
- [A/A split](#quick-start-aa)
- [Homogeneity check](#quick-start-homogeneity)
- [Matching](#quick-start-matching)
- [Run the experiments on Spark](#quick-start-spark)

In [1]:
from hypex import AATest, ABTest, HomogeneityTest, Matching
from hypex.dataset import Dataset, FeatureRole, TargetRole, TreatmentRole
from hypex.utils.tutorial_data_creation import gen_control_variates_df

frame = gen_control_variates_df(
    data_size=500, dependent_division=False, random_state=42
)
frame.head()

,X_lag_1,Target_lag_1,X,Treatment,Target
0,6.736209,873.620926,0.808350,1,290.834973
1,2.377433,437.743256,2.653356,1,475.335563
2,5.454544,745.454379,5.764156,1,786.415578
3,3.373274,537.327376,4.235613,1,633.561305
4,0.130072,213.007158,1.734850,0,373.484971


<a id="quick-start-ab"></a>

## Compare a metric in an A/B test

Assign `TreatmentRole` to the group column and `TargetRole` to the outcome. In `summary`, look for the group means, their difference, and the t-test p-value.

In [2]:
ab_roles = {"Treatment": TreatmentRole(), "Target": TargetRole()}
ab_data = Dataset(roles=ab_roles, data=frame)
ab = ABTest().execute(ab_data)
ab.summary

,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value
0,Target,1,450.598284,461.703547,11.105263,2.46456,-32.446453,54.656978,NOT OK,0.61745


<a id="quick-start-aa"></a>

## Try an A/A split

Pass the historical metric. HypEx creates candidate groups and returns the selected assignment in `best_split`. The `split` column contains the labels `control` and `test_1`.

In [3]:
aa_data = Dataset(
    roles={"Target_lag_1": TargetRole()},
    data=frame[["Target_lag_1"]],
)
aa = AATest().execute(aa_data)
aa.best_split.iloc[:5]

,Target_lag_1,split
0,873.620926,test_1
1,437.743256,test_1
2,745.454379,control
3,537.327376,test_1
4,213.007158,control


<a id="quick-start-homogeneity"></a>

## Check existing groups

Compare the historical metric across the existing groups. In `summary`, find the means and the homogeneity test results. If a test detects a difference, examine the baseline imbalance before interpreting the A/B result.

In [4]:
baseline_data = Dataset(
    roles={"Treatment": TreatmentRole(), "Target_lag_1": TargetRole()},
    data=frame,
)
homogeneity = HomogeneityTest().execute(baseline_data)
homogeneity.summary

,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value,KSTest pass,KSTest p-value
0,Target_lag_1,1,425.69856,445.728331,20.029771,4.705154,-23.115404,63.174947,OK,0.363309,OK,0.69518


<a id="quick-start-matching"></a>

## Estimate an effect with matching

In observational data, pretreatment features can help find comparable observations in the other group. Start with the default Mahalanobis distance, which accounts for scale and correlation when several features are present. Here one historical feature is enough to demonstrate the workflow. The `summary` reports average treatment effects for the full sample (ATE), treated units (ATT), and control units (ATC).

The example spells out the default `distance="mahalanobis"`. This call is equivalent to `Matching()`.

In [5]:
matching_data = Dataset(
    roles={
        "Treatment": TreatmentRole(),
        "Target": TargetRole(),
        "X_lag_1": FeatureRole(),
    },
    data=frame,
)
matched = Matching(distance="mahalanobis").execute(matching_data)
matched.summary

,Effect Size,Standard Error,P-value,CI Lower,CI Upper
ATT,11.3687,37.4879,0.7617,-62.1076,84.8450
ATC,16.6568,36.5984,0.6490,-55.0760,88.3895
ATE,14.0339,33.0933,0.6715,-50.8290,78.8967


<a id="quick-start-spark"></a>

## Run the experiments on Spark

You can run this section in a fresh notebook. For local Spark, set `JAVA_HOME` to a Java 17 installation before starting Jupyter.

Start one session with two local threads, create a temporary checkpoint directory, and generate shared input for all four experiments. Two shuffle partitions are enough for this small table. If `PYSPARK_PYTHON` is unset, the example points Spark workers to the current Python interpreter.

In [6]:
import os
import sys
from tempfile import TemporaryDirectory

from IPython.display import display
from pyspark.sql import SparkSession

from hypex import AATest, ABTest, HomogeneityTest, Matching
from hypex.dataset import Dataset, FeatureRole, TargetRole, TreatmentRole
from hypex.utils.tutorial_data_creation import gen_control_variates_df

os.environ.setdefault("PYSPARK_PYTHON", sys.executable)

checkpoints = TemporaryDirectory(prefix="hypex-quick-start-")
spark = (
    SparkSession.builder.master("local[2]")
    .appName("HypEx Quick Start")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)
spark.sparkContext.setCheckpointDir(checkpoints.name)

spark_frame = spark.createDataFrame(
    gen_control_variates_df(data_size=500, dependent_division=False, random_state=42)
)
print("Spark version:", spark.version)

Spark version: 3.5.1


The version number confirms that the session started. To use your own data, pass a Spark DataFrame, such as the result of `spark.read.parquet(...)`, and assign the roles from the example you want to run. HypEx infers the backend from the input DataFrame. Column roles, the `execute(...)` call, and the reports work the same way as on pandas.

### A/B test on Spark

Compare `Target` across the groups in `Treatment`. In the full `summary`, find the difference in means and the t-test p-value.

In [7]:
spark_ab_data = Dataset(
    roles={"Treatment": TreatmentRole(), "Target": TargetRole()},
    data=spark_frame.select("Treatment", "Target"),
)
print("Backend:", spark_ab_data.backend_type.value)
spark_ab = ABTest().execute(spark_ab_data)
display(spark_ab.summary)

Backend: spark


,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value
0,Target,1,450.598284,461.703547,11.105263,2.46456,-32.446453,54.656978,NOT OK,0.61745


`Backend: spark` confirms the backend. In this example, `difference` is about 11.11 and `TTest p-value` is about 0.617. Both match the pandas results.

### A/A test on Spark

Pass the historical metric `Target_lag_1`. Set `random_states=[42]` to get a single reproducible split, then inspect the first five assignments in `best_split`.

In [8]:
spark_aa_data = Dataset(
    roles={"Target_lag_1": TargetRole()},
    data=spark_frame.select("Target_lag_1"),
)
spark_aa = AATest(random_states=[42]).execute(spark_aa_data)
display(spark_aa.best_split.iloc[:5])

,Target_lag_1,split
0,873.620926,control
1,437.743256,test_1
2,745.454379,control
3,537.327376,control
4,213.007158,control


The `split` column contains `control` and `test_1`. Spark and pandas use different hashing algorithms, so the same seed may assign different users to each group.

### Homogeneity check on Spark

Compare the historical metric `Target_lag_1` in the existing groups and display the full `summary`.

In [9]:
spark_baseline_data = Dataset(
    roles={"Treatment": TreatmentRole(), "Target_lag_1": TargetRole()},
    data=spark_frame.select("Treatment", "Target_lag_1"),
)
spark_homogeneity = HomogeneityTest().execute(spark_baseline_data)
display(spark_homogeneity.summary)

,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value,KSTest pass,KSTest p-value
0,Target_lag_1,1,425.69856,445.728331,20.029771,4.705154,-23.115404,63.174947,OK,0.363309,OK,0.714869


In the `Target_lag_1` row, `TTest p-value` corresponds to the comparison of means and `KSTest p-value` to the comparison of distributions.

### Matching on Spark

Use `X_lag_1` as the feature for finding neighbors. `Matching()` uses Mahalanobis distance by default. The full `summary` contains estimates and confidence intervals for ATT, ATC, and ATE.

In [10]:
spark_matching_data = Dataset(
    roles={
        "Treatment": TreatmentRole(),
        "Target": TargetRole(),
        "X_lag_1": FeatureRole(),
    },
    data=spark_frame.select("Treatment", "Target", "X_lag_1"),
)
spark_matched = Matching().execute(spark_matching_data)
display(spark_matched.summary)

,Effect Size,Standard Error,P-value,CI Lower,CI Upper
ATT,11.3687,37.4879,0.7617,-62.1076,84.8450
ATC,16.6568,36.5984,0.6490,-55.0760,88.3896
ATE,14.0339,33.0933,0.6715,-50.8289,78.8968


To read the estimate for the full sample, find the `ATE` row. `Effect Size` gives the estimate. `CI Lower` and `CI Upper` give the interval bounds.

Stop the local session and remove its temporary directory after running the examples. See the [Spark tutorial](quick_tutorials/SparkBackend.ipynb) for more scenarios.

In [11]:
spark.stop()
checkpoints.cleanup()
print("Spark session stopped")

Spark session stopped


Choose a notebook from [Quick Tutorials](quick_tutorials/README.md) to configure an experiment and explore its results.